# Capítulo 5 · Medidas de frequência em saúde: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática.

A **Parte A** percorre o caso do capítulo, o relatório anual da secretaria de saúde de um município de 200.000 habitantes, pelas perguntas da secretaria; aparecem só as seções com cálculo (5.4, 5.9, 5.10 e 5.11). A **Parte B** traz a resolução escrita de todos os exercícios do capítulo (1 a 14; os enunciados estão no livro) e, nos exercícios cuja resolução precisa de cálculo (13 e 14), o código e o resultado. O código é Python 3 com pandas. Execute as células em ordem.

## Parte A. O caso

**Dados.** `data/diabetes_cohort.csv`: 1.200 adultos acompanhados na atenção primária em 2025, uma linha por pessoa (`person_id`, `entered`, `exited`, `diagnosed`, `exit_reason`). `data/mortality.csv`: população e óbitos por faixa etária (7 faixas) no município A e no município vizinho B. `data/surveillance.csv`: 1.000 casos notificados de uma arbovirose (`case_id`, `month`, `age_group`, `district`, `hospitalized`, `died`), usado pelo Exercício 13. As contagens do programa de vacinação e do hospital estão escritas nas células. Cada célula traz o número da seção do livro a que pertence.

### 5.4 Incidência

*Com que frequência aparecem casos novos de diabetes na coorte?*

In [1]:
# 1. Ler a tabela de seguimento: uma linha por pessoa, com as datas de entrada e de saída e se a saída foi um diagnóstico
import pandas as pd

cohort = pd.read_csv("data/diabetes_cohort.csv", parse_dates=["entered", "exited"])

# 2. Seguimento de cada pessoa em dias completos e as pessoas-ano (a soma dos dias dividida por 365,25)
days = (cohort["exited"] - cohort["entered"]).dt.days
person_years = days.sum() / 365.25
new_cases = cohort["diagnosed"].sum()
print("Pessoas:", len(cohort), "; casos novos:", new_cases, "; pessoas-ano:", round(person_years, 1))

# 3. A fração de diagnósticos observada (casos divididos por pessoas) e a taxa de incidência (casos divididos pelo tempo-pessoa)
print("Fração de diagnósticos observada:", round(100 * new_cases / len(cohort), 2), "%")
print("Taxa de incidência:", round(1000 * new_cases / person_years, 1), "por 1.000 pessoas-ano")

Pessoas: 1200 ; casos novos: 31 ; pessoas-ano: 1135.8
Fração de diagnósticos observada: 2.58 %
Taxa de incidência: 27.3 por 1.000 pessoas-ano


### 5.9 Cobertura do programa

*A cobertura da vacina DTP foi mesmo de 104%?*

In [2]:
# 1. As contagens do programa, cada uma mantida com o seu próprio denominador
doses = 2496
cohort_size = 2400
outside = 138
duplicated = 61
children_3 = 600
valid = doses - outside - duplicated

# 2. A estimativa administrativa: doses divididas por crianças
print("Cobertura administrativa (doses sobre crianças):", round(100 * doses / cohort_size, 1), "%")

# 3. A cobertura de doses auditada: doses válidas sobre as três doses de que cada criança precisa
print("Cobertura de doses auditada (doses válidas sobre 3 doses por criança):", round(100 * valid / (3 * cohort_size), 1), "%")

# 4. A proporção de crianças com três doses
print("Crianças com três doses:", round(100 * children_3 / cohort_size, 1), "%")

# 5. A estimativa do inquérito: 98 vacinadas entre 420 crianças
print("Estimativa do inquérito:", round(100 * 98 / 420, 1), "%")

Cobertura administrativa (doses sobre crianças): 104.0 %
Cobertura de doses auditada (doses válidas sobre 3 doses por criança): 31.9 %
Crianças com três doses: 25.0 %
Estimativa do inquérito: 23.3 %


### 5.10 Taxas brutas e padronizadas

*A mortalidade é mesmo maior no município vizinho?*

In [3]:
# 1. Ler os óbitos e as populações por faixa etária dos municípios A e B
import pandas as pd

m = pd.read_csv("data/mortality.csv")

# 2. Taxas brutas por 1.000: todos os óbitos sobre a população inteira
print("Taxa bruta, município A:", round(1000 * m["deaths_a"].sum() / m["population_a"].sum(), 2), "; município B:", round(1000 * m["deaths_b"].sum() / m["population_b"].sum(), 2))

# 3. Padronização direta: as taxas específicas por idade de cada município aplicadas à população combinada de A e B (o padrão)
rate_a = m["deaths_a"] / m["population_a"]
rate_b = m["deaths_b"] / m["population_b"]
standard = m["population_a"] + m["population_b"]
print("Taxa padronizada, município A:", round(1000 * (rate_a * standard).sum() / standard.sum(), 2), "; município B:", round(1000 * (rate_b * standard).sum() / standard.sum(), 2))

# 4. Padronização indireta: óbitos esperados em B sob as taxas específicas por idade de A e a razão entre óbitos observados e esperados
expected_b = (rate_a * m["population_b"]).sum()
print("Óbitos observados em B:", m["deaths_b"].sum(), "; esperados:", round(expected_b))
print("Razão de mortalidade padronizada de B (observados sobre esperados):", round(m["deaths_b"].sum() / expected_b, 3))

Taxa bruta, município A: 7.56 ; município B: 12.39
Taxa padronizada, município A: 10.39 ; município B: 9.4
Óbitos observados em B: 1859 ; esperados: 2124
Razão de mortalidade padronizada de B (observados sobre esperados): 0.875


### 5.11 Indicadores de serviços hospitalares

*Como são usados os leitos do hospital de 300 leitos?*

In [4]:
# 1. O ano de um hospital de 300 leitos: leitos, dias, leitos-dia ocupados e internações encerradas
beds = 300
days = 365
occupied_bed_days = 91980
ended_stays = 12264

# 2. Os três indicadores
occupancy = occupied_bed_days / (beds * days)
period_stay = occupied_bed_days / ended_stays
turnover = ended_stays / beds
print("Taxa de ocupação:", round(100 * occupancy, 1), "%")
print("Indicador de permanência do período:", round(period_stay, 1), "dias")
print("Giro:", round(turnover, 1), "internações por leito")

# 3. O giro pela identidade: dias vezes ocupação divididos pelo indicador de permanência do período
print("Giro pela identidade:", round(days * occupancy / period_stay, 1), "internações por leito")

Taxa de ocupação: 84.0 %
Indicador de permanência do período: 7.5 dias
Giro: 40.9 internações por leito
Giro pela identidade: 40.9 internações por leito


## Parte B. Soluções dos exercícios

### 5.14 Exercício 1. Três medidas, três denominadores

Moradores com 80 anos ou mais: $6.000/120.000=5\%$, uma proporção da população.
Residentes em instituições de longa permanência: $120/6.000=2\%$ do grupo mais
velho, outra proporção. A mortalidade anual por influenza nesse grupo é
$42/6.000\times100.000=700$ por 100.000 pessoas-ano, usando a população do grupo
como aproximação da pessoa-tempo anual. A terceira medida estima uma taxa de
eventos; seu denominador de tempo está implícito no cálculo com a população anual.

### 5.14 Exercício 2. Um resultado de rastreamento

$84/2.800=3\%$ é a prevalência de doença antes não detectada no momento do
rastreamento: conta os casos existentes, de qualquer duração, entre as mulheres
examinadas. Não diz nada sobre quantas mulheres desenvolvem a condição a cada
ano, o que seria uma incidência e exigiria acompanhar ao longo do tempo mulheres
livres da condição. Um programa de rastreamento detecta, em sua primeira rodada,
os casos acumulados ao longo de muitos anos.

### 5.14 Exercício 3. Incidência acumulada e taxa

A fração de diagnósticos observada é $18/950=1{,}89\%$. Não é o risco em um ano
plenamente observado, pois 40 profissionais de enfermagem têm desfechos
desconhecidos depois de saírem. O risco está entre $18/950=1{,}89\%$ e
$(18+40)/950=6{,}11\%$ se os diagnósticos registrados estiverem corretos e esses
forem os únicos desfechos ausentes.
Quanto à pessoa-tempo, 892 profissionais contribuem com um ano completo e os 58
que saíram ou desenvolveram dor contribuem com meio ano cada um: $892+29=921$
pessoas-ano. A taxa de incidência é $18/921\times1.000=19{,}5$ por 1.000
pessoas-ano. A porcentagem e a taxa têm unidades diferentes; uma não é
simplesmente uma versão maior da outra.

### 5.14 Exercício 4. Prevalência a partir da incidência e da duração

Para X, $ID=0{,}032\times8=0{,}256$, enquanto $ID/(1+ID)=0{,}256/1{,}256=20{,}4\%$.
Para Y, $ID=0{,}004\times4=0{,}016$, enquanto $0{,}016/1{,}016=1{,}57\%$.
A aproximação para condições raras dá 25,6% e 1,6%, respectivamente; é muito
menos precisa para X. Sob o modelo de equilíbrio declarado, X afeta cerca de
13 vezes mais pessoas em um momento e surge oito vezes mais rapidamente entre as
pessoas livres da condição. A prevalência descreve o volume potencial de cuidado
continuado, mas as necessidades do serviço dependem também da gravidade da doença
e das necessidades de cuidado.

### 5.14 Exercício 5. Um jantar de casamento

Taxa de ataque: $58/240=24{,}2\%$. Proporção de casos internados:
$12/58=20{,}7\%$, uma medida de gravidade entre os casos. Taxa de ataque entre os
que comeram a entrada: $27/95=28{,}4\%$. O último número só ajuda a identificar a
causa quando comparado com a taxa de ataque entre os que não comeram a entrada,
$(58-27)/(240-95)=31/145=21{,}4\%$; essa comparação é uma medida de associação,
desenvolvida no Capítulo 7.

### 5.14 Exercício 6. Três denominadores em uma cidade

Letalidade: $4/100=4\%$, óbitos sobre casos; a gravidade da doença.
Mortalidade específica por causa: $(4/3.000)\times100.000\approx133$ por 100.000,
óbitos sobre a população; a taxa anual de óbitos atribuídos a ela na cidade.
Mortalidade proporcional: $4/26=15{,}4\%$ de todos os óbitos; a parcela dos óbitos
da cidade atribuível à doença, útil para prioridades, mas não um risco.

### 5.14 Exercício 7. Mortalidade infantil e neonatal

$3{,}8/18{,}2=20{,}9\%$ dos óbitos infantis ocorreram antes de 28 dias completos.
Os óbitos neonatais fazem parte dos óbitos infantis: o numerador da taxa neonatal
está contido no numerador da taxa infantil, e ambas compartilham o denominador de
nascidos vivos.

### 5.14 Exercício 8. Letalidade durante uma epidemia

A divisão dá 7,4%, mas o enunciado não nos informa quantos casos completaram uma
janela de seguimento comum. Se alguns desfechos ainda estão pendentes, óbitos
adicionais podem elevar a proporção final entre esses mesmos 1.000 casos. Os 27
óbitos recentes não provam, por si, quantos desfechos continuam sem resolução.
Acompanhe uma coorte de casos definida até o desfecho especificado e verifique
que todos os óbitos pertencem a esses casos. A detecção preferencial de doença
grave pode fazer a letalidade de casos superar a letalidade da infecção, porque
as infecções leves e assintomáticas não constam do denominador de casos. A
magnitude, e se ambos os vieses estão presentes, não pode ser determinada apenas
com as contagens fornecidas.

### 5.14 Exercício 9. Três coberturas

Administrativa: $1.880/1.750=107{,}4\%$. Corrigida:
$(1.880-120-45)/1.750=98{,}0\%$. De inquérito: $322/360=89{,}4\%$. Um parágrafo
adequado: “A cobertura administrativa da primeira dose foi de 107,4% (1.880
doses para uma coorte estimada em 1.750). Depois de retiradas 120 doses aplicadas
a não residentes e 45 registros duplicados, a cobertura foi de 98,0%. Um
inquérito domiciliar com 360 lactentes encontrou 89,4% de vacinados. O registro
e a estimativa populacional estão sendo revisados.”

### 5.14 Exercício 10. Padronização de dois distritos

Taxas brutas: A, $140/50.000=2{,}80$ por 1.000; B, $298/50.000=5{,}96$.
Taxas específicas por idade, por 1.000: A, 1,00 abaixo de 65 anos e 10,0 aos 65
anos ou mais; B, 1,40 e 9,00. População padrão: 60.000 abaixo de 65 anos e 40.000
aos 65 anos ou mais. Taxas padronizadas:
A, $(1{,}00\times60.000+10{,}0\times40.000)/100.000=4{,}60$;
B, $(1{,}40\times60.000+9{,}00\times40.000)/100.000=4{,}44$. Óbitos esperados em B
com as taxas de A: $0{,}001\times20.000+0{,}010\times30.000=320$; $SMR=298/320=0{,}93$.
Após a padronização, A tem a mortalidade ligeiramente maior; a taxa bruta maior de
B reflete sua população mais velha.

### 5.14 Exercício 11. Um hospital de 150 leitos

Ocupação: $51.100/(150\times365)\approx93{,}3\%$. Indicador de permanência no
período: $51.100/6.570\approx7{,}78$ dias. Giro: $6.570/150=43{,}8$ internações por
leito. O uso de quantidades não arredondadas verifica a identidade exatamente.
Esses agregados não fornecem a duração média completa das internações com alta em
2025. Em 2026, a enfermaria fechada retira $30\times120=3.600$ leitos-dia do
denominador da ocupação, que passa a ser $150\times365-3.600=51.150$ leitos-dia
disponíveis.

### 5.14 Exercício 12. Chance (odds) e proporções

$0{,}15/0{,}85=0{,}176$. Uma chance (odds) de 1,5 dá uma proporção de
$1{,}5/2{,}5=0{,}60$. Uma chance (odds) de 3 dá $3/4=0{,}75$: três quartos dos
pacientes foram reinternados. A chance (odds) excede a proporção pelo fator
$1/(1-p)$, que é menor que 1,05 quando $p<1/21\approx0{,}04762$. Para $p>0$, a
diferença relativa é $[p/(1-p)-p]/p=p/(1-p)$, de modo que o limiar é cerca de
4,76%, e não exatamente 5%.

### 5.14 Exercício 13. Incidência por distrito

Norte: 552 casos entre 110.000, 501,8 por 100.000; sul: 448 entre 90.000, 497,8
por 100.000. As incidências são praticamente iguais, embora o norte tenha mais
casos. Letalidade entre os casos internados: norte, 2 de 51, 3,9%; sul, 1 de 42,
2,4%. São números pequenos, e a diferença não pode ser interpretada sem os
métodos de capítulos posteriores.

In [5]:
# 1. Incidência por 100.000 por distrito: os casos de cada distrito divididos pela sua própria população
cases = pd.read_csv("data/surveillance.csv")
population = {"north": 110000, "south": 90000}
by_district = cases.groupby("district").size()
print("Incidência por 100.000 por distrito:", (1e5 * by_district / pd.Series(population)).round(1).to_dict())

# 2. Letalidade (%) entre os casos hospitalizados de cada distrito
admitted = cases[cases["hospitalized"]].groupby("district")
print("Letalidade entre os casos hospitalizados, % por distrito:", (100 * admitted["died"].sum() / admitted.size()).round(1).to_dict())

Incidência por 100.000 por distrito: {'north': 501.8, 'south': 497.8}
Letalidade entre os casos hospitalizados, % por distrito: {'north': 3.9, 'south': 2.4}


### 5.14 Exercício 14. Um padrão diferente

Com a própria população de A como padrão, a taxa padronizada de A é igual à sua
taxa bruta, 7,56 por 1.000, e a de B passa a ser 7,16. A ordenação não muda, A
maior que B, mas os dois números diferem dos obtidos com o padrão combinado, 10,39
e 9,40. Uma taxa padronizada é um recurso de comparação sob um padrão declarado;
descreve a taxa hipotética sob a distribuição etária declarada. A ordenação atual
não muda, mas um padrão diferente pode mudar uma ordenação quando as curvas das
taxas específicas por idade se cruzam.

In [6]:
# 1. Padronização com a população do município A como padrão
standard = m["population_a"]
rate_a = m["deaths_a"] / m["population_a"]
rate_b = m["deaths_b"] / m["population_b"]
print("Taxa padronizada com A como padrão - município A:", round(1000 * (rate_a * standard).sum() / standard.sum(), 2), "; município B:", round(1000 * (rate_b * standard).sum() / standard.sum(), 2))

Taxa padronizada com A como padrão - município A: 7.56 ; município B: 7.16


---
*Dados sintéticos (`generate_data.py`, sementes 5 e 51). Valores apenas didáticos; nada aqui é estimativa para um município real.*